In [1]:
#Imports
import pandas as pd
import matplotlib.pyplot as plt
from ipumspy import readers
import pandas as pd
import os

In [ ]:

# Define the file paths
ddi_file_path = r"C:\Users\linds\OneDrive\Documents\Unidos\cps_00007.xml"
data_file_path = r"C:\Users\linds\OneDrive\Documents\Unidos\cps_00007.dat.gz"
# Check if files exist
print(f"DDI file exists: {os.path.exists(ddi_file_path)}")
print(f"Data file exists: {os.path.exists(data_file_path)}")
if os.path.exists(data_file_path):
    print(f"Data file size: {os.path.getsize(data_file_path) / (1024**3):.2f} GB")

# Read the DDI codebook
print("Reading DDI codebook...")
ddi_codebook = readers.read_ipums_ddi(ddi_file_path)

# Read the microdata into a pandas DataFrame
print("Reading microdata (this may take a while for large files)...")
ipums_df = readers.read_microdata(ddi_codebook, data_file_path)

# Display the first few rows
print(f"\nDataFrame shape: {ipums_df.shape}")
print(ipums_df.head())

DDI file exists: True
Data file exists: True
Data file size: 0.01 GB
Reading DDI codebook...
Reading microdata (this may take a while for large files)...


c:\Users\linds\repos\Unidos\venv\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


# Follow the Borjas Methodology for Calculating Likely Unauthorized Immigrants

Note: Full sample size for (2024-2025) 159,301; in Borjas paper (2012-2013 comparison file), full sample size is 191,768.

In [ ]:
#Sample Restrictions
#Drop all except ages 20-64.
ipums_df = ipums_df[ipums_df['AGE'].between(20, 64, inclusive='both')]

Step 1: Restrict to Foreign-Born Population

In [ ]:

foreign_born_df = ipums_df[pd.to_numeric(ipums_df.get('BPL'), errors='coerce') != 9900]  # Assuming 'BPL' is the birthplace variable and 9900 indicates native-born
#44,952 (21%) vs 28,074 (22%) in Borjas sample.

Step 2: Assign "likely legal" if someone immigrated before 1980, or was born in Cuba and immigrated before 2017.

In [ ]:
#Step 2: Assign "likely legal" variable if YRIMMIG <1980 (7), or born in Cuba before 2017
#Note: ~3,380 are True for this. verify this number.
foreign_born_df['likely_legal'] = foreign_born_df['YRIMMIG'] <= 7
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | ((pd.to_numeric(foreign_born_df.get('BPL'), errors='coerce') == 25000) & (pd.to_numeric(foreign_born_df.get('YRIMMIG'), errors='coerce') >= 53))



Step 3: Assign likely legal if receiving public benefits, such as social security income.

In [ ]:
#Step 3: Replace likely_legal=true if INCSSI or INCSS>0
#Note: Updates count to 6,462
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('INCSSI', 0), errors='coerce').fillna(0) > 0) | (pd.to_numeric(foreign_born_df.get('INCSS', 0), errors='coerce').fillna(0) > 0)

#6958--true


Step 4: Assign likely legal if receiving public housing assistance.

In [ ]:
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('PUBHOUS', 0), errors='coerce').fillna(0) > 1)

#true--7359

Step 5: Assign likely legal if they are a veteran.

In [ ]:
#Step 5: Update to legal if veteran
#update to 7,035
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('VETSTAT', 0), errors='coerce') == 2)



Step 6: Assign likely legal if receiving Medicaid or Medicare.

In [ ]:
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('HIMCAIDNW', 0), errors='coerce') ==2)
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('HIMCARENW', 0), errors='coerce') ==2)


Sstep 7: Assign likely legal if their status is reported as a citizen.

In [ ]:
#step 7: Update to likely legal if naturalized citizen
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('CITIZEN', 0), errors='coerce') <=4)

#False: 13,287 (38%) out of 34952 vs 13,566 out of 41,604 (32%)--this is a distinct difference. 


Step 8: Assign likely legal if they are a federal employee.

In [ ]:
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('CLASSWKR', 0), errors='coerce') ==25)
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('CLASSWKR', 0), errors='coerce') ==26)
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('CLASSWKR', 0), errors='coerce') ==27)
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | (pd.to_numeric(foreign_born_df.get('CLASSWKR', 0), errors='coerce') ==28)


Step 9: Assign likely legal if they are in fields requiring licenses, following Borjas definitions.

In [ ]:
#step 9: OCC is licensed/regulated groups
occ_numeric = pd.to_numeric(foreign_born_df.get('OCC', 0), errors='coerce').fillna(0)
in_range = (
    ((occ_numeric >= 2200) & (occ_numeric <= 3540)) |  # Healthcare Practitioners
    ((occ_numeric >= 3700) & (occ_numeric <= 3950)) |  # Protective Service
    ((occ_numeric >= 2100) & (occ_numeric <= 2160)) |  # Legal
    ((occ_numeric >= 2300) & (occ_numeric <= 2550)) |  # Education
    ((occ_numeric >= 9030) & (occ_numeric <= 9040)) |  # Aviation/Transport
    ((occ_numeric >= 1300) & (occ_numeric <= 1560))    # Engineering/Arch
)
foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | in_range

#11,882 (34%)

Step 11: H1B Visa Designation

In [ ]:
# step 11: H1B-related likely legal designation
educ_numeric = pd.to_numeric(foreign_born_df.get('EDUC', 0), errors='coerce').fillna(0)
yrimmig_numeric = pd.to_numeric(foreign_born_df.get('YRIMMIG', foreign_born_df.get('yrimmig', 0)), errors='coerce').fillna(0)

h1b_likely_legal = (educ_numeric >= 111) | (yrimmig_numeric >= 56)
foreign_born_df.loc[:, 'likely_legal'] = foreign_born_df['likely_legal'] | h1b_likely_legal

# EDUC 111+ or YRIMMIG 56+ marked likely legal

Step 10: Assign likely legal if their spouse (in accordance with above criteria) is assigned as likely legal.

In [ ]:
#step 10: If your spouse (identified by SPLOC) is likely legal, you are too
# Build a dictionary mapping (CPSID, PERNUM) to legal status
legal_map = {}
for idx, row in foreign_born_df.iterrows():
    legal_map[(row['CPSID'], row['PERNUM'])] = row['likely_legal']

# Check if spouse is legal
def is_spouse_legal(row):
    if pd.isna(row['SPLOC']) or row['SPLOC'] == 0:
        return False
    return legal_map.get((row['CPSID'], row['SPLOC']), False)

foreign_born_df['likely_legal'] = foreign_born_df['likely_legal'] | foreign_born_df.apply(is_spouse_legal, axis=1)

#10,363 (30%) vs (32%)

# Occupational Distribution of "Likely Legal" vs "Likely Unauthorized"

State of California, to start

In [ ]:
# Create occupation description mapping based on OCC2010 codes
occupation_ranges = [
    (10, 430, "Management in Business, Science, and Arts"),
    (500, 730, "Business Operations Specialists"),
    (800, 950, "Financial Specialists"),
    (1000, 1240, "Computer and Mathematical"),
    (1300, 1540, "Architecture and Engineering"),
    (1550, 1560, "Technicians"),
    (1600, 1980, "Life, Physical, and Social Science"),
    (2000, 2060, "Community and Social Services"),
    (2100, 2150, "Legal"),
    (2200, 2550, "Education, Training, and Library"),
    (2600, 2920, "Arts, Design, Entertainment, Sports, and Media"),
    (3000, 3540, "Healthcare Practitioners and Technicians"),
    (3600, 3650, "Healthcare Support"),
    (3700, 3950, "Protective Service"),
    (4000, 4150, "Food Preparation and Serving"),
    (4200, 4250, "Building and Grounds Cleaning and Maintenance"),
    (4300, 4650, "Personal Care and Service"),
    (4700, 4965, "Sales and Related"),
    (5000, 5940, "Office and Administrative Support"),
    (6005, 6130, "Farming, Fisheries, and Forestry"),
    (6200, 6765, "Construction"),
    (6800, 6940, "Extraction"),
    (7000, 7630, "Installation, Maintenance, and Repair"),
    (7700, 8965, "Production"),
    (9000, 9750, "Transportation and Material Moving"),
    (9800, 9830, "Military"),
    (9920, 9920, "No Occupation")
]

def map_occupation(occ_code):
    """Map OCC2010 code to occupation description."""
    if pd.isna(occ_code) or occ_code == 0:
        return "Unknown"
    for start, end, desc in occupation_ranges:
        if start <= occ_code <= end:
            return desc
    return "Unknown"

foreign_born_df['occupation_desc'] = foreign_born_df['OCC2010'].apply(map_occupation)

All Foreign Born

In [ ]:
occupation_counts = foreign_born_df['occupation_desc'].value_counts().sort_values(ascending=True)
occupation_counts = occupation_counts[occupation_counts.index != 'Unknown'].sort_values(ascending=True)
# Create a horizontal bar chart for better readability
fig, ax = plt.subplots(figsize=(12, 10))
occupation_counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Distribution of Occupations (Foreign-Born Population)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nTotal records: {len(foreign_born_df)}")
print(f"Number of occupation categories: {len(occupation_counts)}")
print(f"\nTop 5 most common occupations:")
print(foreign_born_df['occupation_desc'].value_counts().head())

Likely Legal

In [ ]:
df_likely_legal = foreign_born_df[foreign_born_df['likely_legal']]

In [ ]:

# Count the distribution of occupation descriptions in California
occupation_counts = df_likely_legal['occupation_desc'].value_counts().sort_values(ascending=True)
occupation_counts = occupation_counts[occupation_counts.index != 'Unknown'].sort_values(ascending=True)

# Create a horizontal bar chart for better readability
fig, ax = plt.subplots(figsize=(12, 10))
occupation_counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Distribution of Occupations (Foreign-Born Population, Likely Legal)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nTotal records: {len(df_likely_legal)}")
print(f"Number of occupation categories: {len(occupation_counts)}")
print(f"\nTop 5 most common occupations:")
print(df_likely_legal['occupation_desc'].value_counts().head())

Likely Unauthorized

In [ ]:
#Create unauthorized df
df_likely_unauthorized = foreign_born_df[~foreign_born_df['likely_legal']]
# Count the distribution of occupation descriptions in California
occupation_counts = df_likely_unauthorized['occupation_desc'].value_counts().sort_values(ascending=True)
occupation_counts = occupation_counts[occupation_counts.index != 'Unknown'].sort_values(ascending=True)

# Create a horizontal bar chart for better readability
fig, ax = plt.subplots(figsize=(12, 10))
occupation_counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Distribution of Occupations (Foreign-Born Population, Likely Unauthorized)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nTotal records: {len(df_likely_unauthorized)}")
print(f"Number of occupation categories: {len(occupation_counts)}")
print(f"\nTop 5 most common occupations:")
print(df_likely_unauthorized['occupation_desc'].value_counts().head())

In [ ]:
df_h1b_likely_legal = foreign_born_df[h1b_likely_legal].copy()

occupation_counts = df_h1b_likely_legal['occupation_desc'].value_counts().sort_values(ascending=True)
occupation_counts = occupation_counts[occupation_counts.index != 'Unknown'].sort_values(ascending=True)
# Create a horizontal bar chart for better readability
fig, ax = plt.subplots(figsize=(12, 10))
occupation_counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Distribution of Occupations (Foreign-Born Population)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nTotal records: {len(df_h1b_likely_legal)}")
print(f"Number of occupation categories: {len(occupation_counts)}")
print(f"\nTop 5 most common occupations:")
print(df_h1b_likely_legal['occupation_desc'].value_counts().head())

# Detailed Occupations

In [ ]:
# Create detailed occupation mapping based on OCC2010 codes with individual labels
detailed_occ_mapping = {
    10: "Chief executives and legislators",
    20: "General and Operations Managers",
    40: "Advertising and Promotions Managers",
    50: "Marketing and Sales Managers",
    60: "Public Relations and Fundraising Managers",
    100: "Administrative Services Managers",
    110: "Computer and Information Systems Managers",
    120: "Financial Managers",
    135: "Compensation and benefits managers",
    136: "Human Resources Managers",
    137: "Training and development managers",
    140: "Industrial Production Managers",
    150: "Purchasing Managers",
    160: "Transportation, Storage, and Distribution Managers",
    205: "Farmers, Ranchers, and Other Agricultural Managers",
    220: "Construction Managers",
    230: "Education Administrators",
    300: "Architectural and Engineering Managers",
    310: "Food Service Managers",
    330: "Gaming Managers",
    340: "Lodging Managers",
    350: "Medical and Health Services Managers",
    360: "Natural Sciences Managers",
    410: "Property, Real Estate, and Community Association Managers",
    420: "Social and Community Service Managers",
    425: "Emergency management directors",
    430: "Miscellaneous managers, including funeral service managers and postmasters and mail superintendents",
    500: "Agents and Business Managers of Artists, Performers, and Athletes",
    510: "Buyers and Purchasing Agents, Farm Products",
    520: "Wholesale and Retail Buyers, Except Farm Products",
    530: "Purchasing Agents, Except Wholesale, Retail, and Farm Products",
    540: "Claims Adjusters, Appraisers, Examiners, and Investigators",
    565: "Compliance Officers",
    600: "Cost Estimators",
    630: "Human Resources Workers",
    640: "Compensation, benefits, and job analysis specialists",
    650: "Training and development specialists",
    700: "Logisticians",
    710: "Management Analysts",
    725: "Meeting, Convention, and Event Planners",
    726: "Fundraisers",
    735: "Market Research Analysts and Marketing Specialists",
    740: "Business Operations Specialists, All Other",
    800: "Accountants and Auditors",
    810: "Appraisers and Assessors of Real Estate",
    820: "Budget Analysts",
    830: "Credit Analysts",
    840: "Financial Analysts",
    850: "Personal Financial Advisors",
    860: "Insurance Underwriters",
    900: "Financial Examiners",
    910: "Credit Counselors and Loan Officers",
    930: "Tax Examiners and Collectors, and Revenue Agents",
    940: "Tax Preparers",
    950: "Financial Specialists, All Other",
    1005: "Computer and information research scientists",
    1006: "Computer Systems Analysts",
    1007: "Information security analysts",
    1010: "Computer Programmers",
    1020: "Software Developers, Applications and Systems Software",
    1030: "Web Developers",
    1050: "Computer Support Specialists",
    1060: "Database Administrators",
    1105: "Network and Computer Systems Administrators",
    1106: "Computer network architects",
    1107: "Computer occupations, all other",
    1200: "Actuaries",
    1220: "Operations Research Analysts",
    1240: "Miscellaneous mathematical science occupations, including mathematicians and statisticians",
    1300: "Architects, Except Naval",
    1310: "Surveyors, Cartographers, and Photogrammetrists",
    1320: "Aerospace Engineers",
    1340: "Biomedical and agricultural engineers",
    1350: "Chemical Engineers",
    1360: "Civil Engineers",
    1400: "Computer Hardware Engineers",
    1410: "Electrical and Electronics Engineers",
    1420: "Environmental Engineers",
    1430: "Industrial Engineers, including Health and Safety",
    1440: "Marine Engineers and Naval Architects",
    1450: "Materials Engineers",
    1460: "Mechanical Engineers",
    1520: "Petroleum, mining and geological engineers, including mining safety engineers",
    1530: "Miscellaneous engineers, including nuclear engineers",
    1540: "Drafters",
    1550: "Engineering Technicians, Except Drafters",
    1560: "Surveying and Mapping Technicians",
    1600: "Agricultural and Food Scientists",
    1610: "Biological Scientists",
    1640: "Conservation Scientists and Foresters",
    1650: "Medical scientists, and life scientists, all other",
    1700: "Astronomers and Physicists",
    1710: "Atmospheric and Space Scientists",
    1720: "Chemists and Materials Scientists",
    1740: "Environmental Scientists and Geoscientists",
    1760: "Physical Scientists, All Other",
    1800: "Economists",
    1820: "Psychologists",
    1840: "Urban and Regional Planners",
    1860: "Miscellaneous social scientists, including survey researchers and sociologists",
    1900: "Agricultural and Food Science Technicians",
    1910: "Biological Technicians",
    1920: "Chemical Technicians",
    1930: "Geological and petroleum technicians, and nuclear technicians",
    1965: "Miscellaneous life, physical, and social science technicians, including social science research assistants",
    2000: "Counselors",
    2010: "Social Workers",
    2015: "Probation officers and correctional treatment specialists",
    2016: "Social and human service assistants",
    2025: "Miscellaneous community and social service specialists, including health educators and community health workers",
    2040: "Clergy",
    2050: "Directors, Religious Activities and Education",
    2060: "Religious Workers, All Other",
    2100: "Lawyers, and judges, magistrates, and other judicial workers",
    2105: "Judicial law clerks",
    2145: "Paralegals and Legal Assistants",
    2160: "Miscellaneous Legal Support Workers",
    2200: "Postsecondary Teachers",
    2300: "Preschool and Kindergarten Teachers",
    2310: "Elementary and Middle School Teachers",
    2320: "Secondary School Teachers",
    2330: "Special Education Teachers",
    2340: "Other Teachers and Instructors",
    2400: "Archivists, Curators, and Museum Technicians",
    2430: "Librarians",
    2440: "Library Technicians",
    2540: "Teacher Assistants",
    2550: "Other Education, Training, and Library Workers",
    2600: "Artists and Related Workers",
    2630: "Designers",
    2700: "Actors",
    2710: "Producers and Directors",
    2720: "Athletes, Coaches, Umpires, and Related Workers",
    2740: "Dancers and Choreographers",
    2750: "Musicians, Singers, and Related Workers",
    2760: "Entertainers and Performers, Sports and Related Workers, All Other",
    2800: "Announcers",
    2810: "News Analysts, Reporters and Correspondents",
    2825: "Public Relations Specialists",
    2830: "Editors",
    2840: "Technical Writers",
    2850: "Writers and Authors",
    2860: "Miscellaneous Media and Communication Workers",
    2900: "Broadcast and sound engineering technicians and radio operators, and media and communication equipment workers, all other",
    2910: "Photographers",
    2920: "Television, Video, and Motion Picture Camera Operators and Editors",
    3000: "Chiropractors",
    3010: "Dentists",
    3030: "Dietitians and Nutritionists",
    3040: "Optometrists",
    3050: "Pharmacists",
    3060: "Physicians and Surgeons",
    3110: "Physician Assistants",
    3120: "Podiatrists",
    3140: "Audiologists",
    3150: "Occupational Therapists",
    3160: "Physical Therapists",
    3200: "Radiation Therapists",
    3210: "Recreational Therapists",
    3220: "Respiratory Therapists",
    3230: "Speech-Language Pathologists",
    3245: "Other therapists, including exercise physiologists",
    3250: "Veterinarians",
    3255: "Registered Nurses",
    3256: "Nurse anesthetists",
    3258: "Nurse practitioners and nurse midwives",
    3260: "Health Diagnosing and Treating Practitioners, All Other",
    3300: "Clinical Laboratory Technologists and Technicians",
    3310: "Dental Hygienists",
    3320: "Diagnostic Related Technologists and Technicians",
    3400: "Emergency Medical Technicians and Paramedics",
    3420: "Health Practitioner Support Technologists and Technicians",
    3500: "Licensed Practical and Licensed Vocational Nurses",
    3510: "Medical Records and Health Information Technicians",
    3520: "Opticians, Dispensing",
    3535: "Miscellaneous Health Technologists and Technicians",
    3540: "Other Healthcare Practitioners and Technical Occupations",
    3600: "Nursing, Psychiatric, and Home Health Aides",
    3610: "Occupational Therapy Assistants and Aides",
    3620: "Physical Therapist Assistants and Aides",
    3630: "Massage Therapists",
    3640: "Dental Assistants",
    3645: "Medical Assistants",
    3646: "Medical transcriptionists",
    3647: "Pharmacy aides",
    3648: "Veterinary assistants and laboratory animal caretakers",
    3649: "Phlebotomists",
    3655: "Healthcare support workers, all other, including medical equipment preparers",
    3700: "First-Line Supervisors of Correctional Officers",
    3710: "First-Line Supervisors of Police and Detectives",
    3720: "First-Line Supervisors of Fire Fighting and Prevention Workers",
    3730: "First-Line Supervisors of Protective Service Workers, All Other",
    3740: "Firefighters",
    3750: "Fire Inspectors",
    3800: "Bailiffs, Correctional Officers, and Jailers",
    3820: "Detectives and Criminal Investigators",
    3840: "Miscellaneous law enforcement workers",
    3850: "Police officers",
    3900: "Animal Control Workers",
    3910: "Private Detectives and Investigators",
    3930: "Security Guards and Gaming Surveillance Officers",
    3940: "Crossing Guards",
    3945: "Transportation security screeners",
    3955: "Lifeguards and Other Recreational, and All Other Protective Service Workers",
    4000: "Chefs and Head Cooks",
    4010: "First-Line Supervisors of Food Preparation and Serving Workers",
    4020: "Cooks",
    4030: "Food Preparation Workers",
    4040: "Bartenders",
    4050: "Combined Food Preparation and Serving Workers, Including Fast Food",
    4060: "Counter Attendants, Cafeteria, Food Concession, and Coffee Shop",
    4110: "Waiters and Waitresses",
    4120: "Food Servers, Nonrestaurant",
    4130: "Miscellaneous food preparation and serving related workers, including dining room and cafeteria attendants and bartender helpers",
    4140: "Dishwashers",
    4150: "Hosts and Hostesses, Restaurant, Lounge, and Coffee Shop",
    4200: "First-Line Supervisors of Housekeeping and Janitorial Workers",
    4210: "First-Line Supervisors of Landscaping, Lawn Service, and Groundskeeping Workers",
    4220: "Janitors and Building Cleaners",
    4230: "Maids and housekeeping cleaners",
    4240: "Pest Control Workers",
    4250: "Grounds Maintenance Workers",
    4300: "First-Line Supervisors of Gaming Workers",
    4320: "First-Line Supervisors of Personal Service Workers",
    4340: "Animal Trainers",
    4350: "Nonfarm Animal Caretakers",
    4400: "Gaming Services Workers",
    4410: "Motion Picture Projectionists",
    4420: "Ushers, Lobby Attendants, and Ticket Takers",
    4430: "Miscellaneous Entertainment Attendants and Related Workers",
    4460: "Embalmers and Funeral Attendants",
    4465: "Morticians, undertakers, and funeral directors",
    4500: "Barbers",
    4510: "Hairdressers, Hairstylists, and Cosmetologists",
    4520: "Miscellaneous Personal Appearance Workers",
    4530: "Baggage Porters, Bellhops, and Concierges",
    4540: "Tour and Travel Guides",
    4600: "Childcare Workers",
    4610: "Personal Care Aides",
    4620: "Recreation and Fitness Workers",
    4640: "Residential Advisors",
    4650: "Personal Care and Service Workers, All Other",
    4700: "First-Line Supervisors of Retail Sales Workers",
    4710: "First-Line Supervisors of Non-Retail Sales Workers",
    4720: "Cashiers",
    4740: "Counter and Rental Clerks",
    4750: "Parts Salespersons",
    4760: "Retail Salespersons",
    4800: "Advertising Sales Agents",
    4810: "Insurance Sales Agents",
    4820: "Securities, Commodities, and Financial Services Sales Agents",
    4830: "Travel Agents",
    4840: "Sales Representatives, Services, All Other",
    4850: "Sales Representatives, Wholesale and Manufacturing",
    4900: "Models, Demonstrators, and Product Promoters",
    4920: "Real Estate Brokers and Sales Agents",
    4930: "Sales Engineers",
    4940: "Telemarketers",
    4950: "Door-to-Door Sales Workers, News and Street Vendors, and Related Workers",
    4965: "Sales and Related Workers, All Other",
    5000: "First-Line Supervisors of Office and Administrative Support Workers",
    5010: "Switchboard Operators, Including Answering Service",
    5020: "Telephone Operators",
    5030: "Communications Equipment Operators, All Other",
    5100: "Bill and Account Collectors",
    5110: "Billing and Posting Clerks",
    5120: "Bookkeeping, Accounting, and Auditing Clerks",
    5130: "Gaming Cage Workers",
    5140: "Payroll and Timekeeping Clerks",
    5150: "Procurement Clerks",
    5160: "Tellers",
    5165: "Financial clerks, all other",
    5200: "Brokerage Clerks",
    5220: "Court, Municipal, and License Clerks",
    5230: "Credit Authorizers, Checkers, and Clerks",
    5240: "Customer Service Representatives",
    5250: "Eligibility Interviewers, Government Programs",
    5260: "File Clerks",
    5300: "Hotel, Motel, and Resort Desk Clerks",
    5310: "Interviewers, Except Eligibility and Loan",
    5320: "Library Assistants, Clerical",
    5330: "Loan Interviewers and Clerks",
    5340: "New Accounts Clerks",
    5350: "Correspondence clerks and order clerks",
    5360: "Human resources assistants, except payroll and timekeeping",
    5400: "Receptionists and Information Clerks",
    5410: "Reservation and Transportation Ticket Agents and Travel Clerks",
    5420: "Information and Record Clerks, All Other",
    5500: "Cargo and Freight Agents",
    5510: "Couriers and Messengers",
    5520: "Dispatchers",
    5530: "Meter Readers, Utilities",
    5540: "Postal Service Clerks",
    5550: "Postal Service Mail Carriers",
    5560: "Postal Service Mail Sorters, Processors, and Processing Machine Operators",
    5600: "Production, Planning, and Expediting Clerks",
    5610: "Shipping, Receiving, and Traffic Clerks",
    5620: "Stock Clerks and Order Fillers",
    5630: "Weighers, Measurers, Checkers, and Samplers, Recordkeeping",
    5700: "Secretaries and Administrative Assistants",
    5800: "Computer Operators",
    5810: "Data Entry Keyers",
    5820: "Word Processors and Typists",
    5840: "Insurance Claims and Policy Processing Clerks",
    5850: "Mail Clerks and Mail Machine Operators, Except Postal Service",
    5860: "Office Clerks, General",
    5900: "Office Machine Operators, Except Computer",
    5910: "Proofreaders and Copy Markers",
    5920: "Statistical Assistants",
    5940: "Miscellaneous office and administrative support workers, including desktop publishers",
    6005: "First-line supervisors of farming, fishing, and forestry workers",
    6010: "Agricultural Inspectors",
    6040: "Graders and Sorters, Agricultural Products",
    6050: "Miscellaneous agricultural workers, including animal breeders",
    6100: "Fishing and hunting workers",
    6120: "Forest and Conservation Workers",
    6130: "Logging Workers",
    6200: "First-Line Supervisors of Construction Trades and Extraction Workers",
    6210: "Boilermakers",
    6220: "Brickmasons, blockmasons, stonemasons, and reinforcing iron and rebar workers",
    6230: "Carpenters",
    6240: "Carpet, Floor, and Tile Installers and Finishers",
    6250: "Cement Masons, Concrete Finishers, and Terrazzo Workers",
    6260: "Construction Laborers",
    6300: "Paving, Surfacing, and Tamping Equipment Operators",
    6320: "Construction equipment operators except paving, surfacing, and tamping equipment operators",
    6330: "Drywall Installers, Ceiling Tile Installers, and Tapers",
    6355: "Electricians",
    6360: "Glaziers",
    6400: "Insulation Workers",
    6420: "Painters and paperhangers",
    6440: "Pipelayers, Plumbers, Pipefitters, and Steamfitters",
    6460: "Plasterers and Stucco Masons",
    6515: "Roofers",
    6520: "Sheet Metal Workers",
    6530: "Structural Iron and Steel Workers",
    6600: "Helpers, Construction Trades",
    6660: "Construction and Building Inspectors",
    6700: "Elevator Installers and Repairers",
    6710: "Fence Erectors",
    6720: "Hazardous Materials Removal Workers",
    6730: "Highway Maintenance Workers",
    6740: "Rail-Track Laying and Maintenance Equipment Operators",
    6765: "Miscellaneous construction workers, including solar photovoltaic installers, septic tank servicers and sewer pipe cleaners",
    6800: "Derrick, rotary drill, and service unit operators, and roustabouts, oil, gas, and mining",
    6820: "Earth Drillers, Except Oil and Gas",
    6830: "Explosives Workers, Ordnance Handling Experts, and Blasters",
    6840: "Mining Machine Operators",
    6940: "Miscellaneous extraction workers, including roof bolters and helpers",
    7000: "First-Line Supervisors of Mechanics, Installers, and Repairers",
    7010: "Computer, Automated Teller, and Office Machine Repairers",
    7020: "Radio and Telecommunications Equipment Installers and Repairers",
    7030: "Avionics Technicians",
    7040: "Electric Motor, Power Tool, and Related Repairers",
    7100: "Electrical and electronics repairers, transportation equipment, and industrial and utility",
    7110: "Electronic Equipment Installers and Repairers, Motor Vehicles",
    7120: "Electronic Home Entertainment Equipment Installers and Repairers",
    7130: "Security and Fire Alarm Systems Installers",
    7140: "Aircraft Mechanics and Service Technicians",
    7150: "Automotive Body and Related Repairers",
    7160: "Automotive Glass Installers and Repairers",
    7200: "Automotive Service Technicians and Mechanics",
    7210: "Bus and Truck Mechanics and Diesel Engine Specialists",
    7220: "Heavy Vehicle and Mobile Equipment Service Technicians and Mechanics",
    7240: "Small Engine Mechanics",
    7260: "Miscellaneous Vehicle and Mobile Equipment Mechanics, Installers, and Repairers",
    7300: "Control and Valve Installers and Repairers",
    7315: "Heating, Air Conditioning, and Refrigeration Mechanics and Installers",
    7320: "Home Appliance Repairers",
    7330: "Industrial and Refractory Machinery Mechanics",
    7340: "Maintenance and Repair Workers, General",
    7350: "Maintenance Workers, Machinery",
    7360: "Millwrights",
    7410: "Electrical Power-Line Installers and Repairers",
    7420: "Telecommunications Line Installers and Repairers",
    7430: "Precision Instrument and Equipment Repairers",
    7510: "Coin, Vending, and Amusement Machine Servicers and Repairers",
    7540: "Locksmiths and Safe Repairers",
    7560: "Riggers",
    7610: "Helpers--Installation, Maintenance, and Repair Workers",
    7630: "Miscellaneous installation, maintenance, and repair workers, including wind turbine service technicians",
    7700: "First-Line Supervisors of Production and Operating Workers",
    7710: "Aircraft Structure, Surfaces, Rigging, and Systems Assemblers",
    7720: "Electrical, Electronics, and Electromechanical Assemblers",
    7730: "Engine and Other Machine Assemblers",
    7740: "Structural Metal Fabricators and Fitters",
    7750: "Miscellaneous Assemblers and Fabricators",
    7800: "Bakers",
    7810: "Butchers and Other Meat, Poultry, and Fish Processing Workers",
    7830: "Food and Tobacco Roasting, Baking, and Drying Machine Operators and Tenders",
    7840: "Food Batchmakers",
    7850: "Food Cooking Machine Operators and Tenders",
    7855: "Food processing workers, all other",
    7900: "Computer Control Programmers and Operators",
    7920: "Extruding and Drawing Machine Setters, Operators, and Tenders, Metal and Plastic",
    7930: "Forging Machine Setters, Operators, and Tenders, Metal and Plastic",
    7940: "Rolling Machine Setters, Operators, and Tenders, Metal and Plastic",
    7950: "Machine tool cutting setters, operators, and tenders, metal and plastic",
    8030: "Machinists",
    8040: "Metal Furnace Operators, Tenders, Pourers, and Casters",
    8100: "Model makers, patternmakers, and molding machine setters, metal and plastic",
    8130: "Tool and Die Makers",
    8140: "Welding, Soldering, and Brazing Workers",
    8220: "Miscellaneous metal workers and plastic workers, including multiple machine tool setters",
    8250: "Prepress Technicians and Workers",
    8255: "Printing Press Operators",
    8256: "Print Binding and Finishing Workers",
    8300: "Laundry and Dry-Cleaning Workers",
    8310: "Pressers, Textile, Garment, and Related Materials",
    8320: "Sewing Machine Operators",
    8330: "Shoe and leather workers",
    8350: "Tailors, Dressmakers, and Sewers",
    8400: "Textile bleaching and dyeing, and cutting machine setters, operators, and tenders",
    8410: "Textile Knitting and Weaving Machine Setters, Operators, and Tenders",
    8420: "Textile Winding, Twisting, and Drawing Out Machine Setters, Operators, and Tenders",
    8450: "Upholsterers",
    8460: "Miscellaneous textile, apparel, and furnishings workers except upholsterers",
    8500: "Cabinetmakers and Bench Carpenters",
    8510: "Furniture Finishers",
    8530: "Sawing Machine Setters, Operators, and Tenders, Wood",
    8540: "Woodworking Machine Setters, Operators, and Tenders, Except Sawing",
    8550: "Miscellaneous woodworkers, including model makers and patternmakers",
    8600: "Power Plant Operators, Distributors, and Dispatchers",
    8610: "Stationary Engineers and Boiler Operators",
    8620: "Water and Wastewater Treatment Plant and System Operators",
    8630: "Miscellaneous Plant and System Operators",
    8640: "Chemical Processing Machine Setters, Operators, and Tenders",
    8650: "Crushing, Grinding, Polishing, Mixing, and Blending Workers",
    8710: "Cutting Workers",
    8720: "Extruding, Forming, Pressing, and Compacting Machine Setters, Operators, and Tenders",
    8730: "Furnace, Kiln, Oven, Drier, and Kettle Operators and Tenders",
    8740: "Inspectors, Testers, Sorters, Samplers, and Weighers",
    8750: "Jewelers and Precious Stone and Metal Workers",
    8760: "Medical, Dental, and Ophthalmic Laboratory Technicians",
    8800: "Packaging and Filling Machine Operators and Tenders",
    8810: "Painting Workers",
    8830: "Photographic Process Workers and Processing Machine Operators",
    8850: "Adhesive Bonding Machine Operators and Tenders",
    8910: "Etchers and Engravers",
    8920: "Molders, Shapers, and Casters, Except Metal and Plastic",
    8930: "Paper Goods Machine Setters, Operators, and Tenders",
    8940: "Tire Builders",
    8950: "Helpers--Production Workers",
    8965: "Miscellaneous production workers, including semiconductor processors",
    9000: "Supervisors of Transportation and Material Moving Workers",
    9030: "Aircraft Pilots and Flight Engineers",
    9040: "Air Traffic Controllers and Airfield Operations Specialists",
    9050: "Flight Attendants",
    9110: "Ambulance Drivers and Attendants, Except Emergency Medical Technicians",
    9120: "Bus Drivers",
    9130: "Driver/Sales Workers and Truck Drivers",
    9140: "Taxi Drivers and Chauffeurs",
    9150: "Motor Vehicle Operators, All Other",
    9200: "Locomotive Engineers and Operators",
    9240: "Railroad Conductors and Yardmasters",
    9260: "Subway, streetcar, and other rail transportation workers",
    9300: "Sailors and marine oilers, and ship engineers",
    9310: "Ship and Boat Captains and Operators",
    9350: "Parking Lot Attendants",
    9360: "Automotive and Watercraft Service Attendants",
    9410: "Transportation Inspectors",
    9415: "Transportation attendants, except flight attendants",
    9420: "Miscellaneous transportation workers, including bridge and lock tenders and traffic technicians",
    9510: "Crane and Tower Operators",
    9520: "Dredge, Excavating, and Loading Machine Operators",
    9560: "Conveyor operators and tenders, and hoist and winch operators",
    9600: "Industrial Truck and Tractor Operators",
    9610: "Cleaners of Vehicles and Equipment",
    9620: "Laborers and Freight, Stock, and Material Movers, Hand",
    9630: "Machine Feeders and Offbearers",
    9640: "Packers and Packagers, Hand",
    9650: "Pumping Station Operators",
    9720: "Refuse and Recyclable Material Collectors",
    9750: "Miscellaneous material moving workers, including mine shuttle car operators, and tank car, truck, and ship loaders",
    9800: "Military Officer Special and Tactical Operations Leaders",
    9810: "First-Line Enlisted Military Supervisors",
    9820: "Military Enlisted Tactical Operations and Air/Weapons Specialists and Crew Members",
    9830: "Military, Rank Not Specified",
    9999: "NIU (Not in Universe)"
}

def map_occupation_detailed(occ_code):
    """Map OCC2010 code to detailed occupation description."""
    if pd.isna(occ_code) or occ_code == 0:
        return "Unknown"
    occ_int = int(occ_code)
    return detailed_occ_mapping.get(occ_int, "Unknown")



All Foreign Born

In [ ]:
foreign_born_df['occupation_desc_detailed'] = foreign_born_df['OCC2010'].apply(map_occupation_detailed)

# Get top 15 occupations (excluding NIU) and create bar chart
occupation_counts = foreign_born_df['occupation_desc_detailed'].value_counts()
occupation_counts_filtered = occupation_counts[occupation_counts.index != "NIU (Not in Universe)"]
top_15_occupations = occupation_counts_filtered.head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
top_15_occupations.plot(kind='barh', ax=ax, color='forestgreen')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Top 15 Occupations - Foreign-Born Population', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal Foreign-Born Records: {len(foreign_born_df)}")
print(f"\nTop 15 Most Common Occupations (Foreign-Born, excluding NIU):")
print(occupation_counts_filtered.head(15))

Likely Legal

In [ ]:
df_likely_legal['occupation_desc_detailed'] = df_likely_legal['OCC2010'].apply(map_occupation_detailed)

# Get top 15 occupations (excluding NIU) and create bar chart
occupation_counts = df_likely_legal['occupation_desc_detailed'].value_counts()
occupation_counts_filtered = occupation_counts[occupation_counts.index != "NIU (Not in Universe)"]
top_15_occupations = occupation_counts_filtered.head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
top_15_occupations.plot(kind='barh', ax=ax, color='forestgreen')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Top 15 Occupations - Foreign-Born Population (Likely Legal)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal Foreign-Born Records: {len(df_likely_legal)}")
print(f"\nTop 15 Most Common Occupations (Foreign-Born, Likely Legal):")
print(occupation_counts_filtered.head(15))

Likely Unauthorized

In [ ]:
df_likely_unauthorized['occupation_desc_detailed'] = df_likely_unauthorized['OCC2010'].apply(map_occupation_detailed)

# Get top 15 occupations (excluding NIU) and create bar chart
occupation_counts = df_likely_unauthorized['occupation_desc_detailed'].value_counts()
occupation_counts_filtered = occupation_counts[occupation_counts.index != "NIU (Not in Universe)"]
top_15_occupations = occupation_counts_filtered.head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
top_15_occupations.plot(kind='barh', ax=ax, color='forestgreen')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Top 15 Occupations - Foreign-Born Population (Likely Unauthorized)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal Foreign-Born Records: {len(df_likely_unauthorized)}")
print(f"\nTop 15 Most Common Occupations (Foreign-Born, Likely Unauthorized):")
print(occupation_counts_filtered.head(15))

# Occupational Distribution of H1B Visa Holders: Validation

In [ ]:
df_h1b_likely_legal = foreign_born_df[h1b_likely_legal].copy()
df_h1b_likely_legal['occupation_desc_detailed'] = df_h1b_likely_legal['OCC2010'].apply(map_occupation_detailed)

# Get top 15 occupations (excluding NIU) and create bar chart
occupation_counts = df_h1b_likely_legal['occupation_desc_detailed'].value_counts()
occupation_counts_filtered = occupation_counts[occupation_counts.index != "NIU (Not in Universe)"]
top_15_occupations = occupation_counts_filtered.head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
top_15_occupations.plot(kind='barh', ax=ax, color='forestgreen')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Top 15 Occupations - Foreign-Born Population H1B Visa', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal Foreign-Born Records: {len(df_h1b_likely_legal)}")
print(f"\nTop 15 Most Common Occupations H1B Visa:")
print(occupation_counts_filtered.head(15))

In [ ]:
h1b_df=pd.read_excel(r"C:\Users\linds\Downloads\Employer Information.xlsx")

In [ ]:
# Get top 15 occupations (excluding NIU) and create bar chart
occupation_counts = h1b_df['Industry (NAICS) Code'].value_counts()
occupation_counts_filtered = occupation_counts[occupation_counts.index != "NIU (Not in Universe)"]
top_15_occupations = occupation_counts_filtered.head(15).sort_values(ascending=True)


fig, ax = plt.subplots(figsize=(12, 8))
top_15_occupations.plot(kind='barh', ax=ax, color='forestgreen')

ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Occupation Description', fontsize=12)
ax.set_title('Top 15 Occupations - H1b Data', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal Foreign-Born Records: {len(h1b_df)}")
print(f"\nTop 15 Most Common Occupations H1B Visa:")
print(occupation_counts_filtered.head(15))
